# LTX-Video image-to-video test for Billionaires Hidden Wife

Use one of your approved storyboard images. This notebook does not download images; it downloads model weights/software only. A Colab GPU is required and free GPU access is not guaranteed. This creates experimental image-to-video motion, not guaranteed dialogue lip-sync or accurate two-character acting.

In [ ]:
# Repair the current Colab dependency conflict: torchao's installed version is missing FqnToConfig.
# LTX-Video can run without torchao quantization; remove it and use a stable Diffusers release.
!pip -q uninstall -y torchao
!pip -q install --upgrade --no-cache-dir "diffusers==0.35.2" "transformers==4.57.1" "accelerate>=1.2.0" "sentencepiece>=0.2.0" safetensors imageio imageio-ffmpeg
print('Packages installed. IMPORTANT: use Runtime → Restart session, then rerun the GPU check and continue from the image upload cell. Do not rerun the install cell after restarting.')


In [ ]:
import torch
from PIL import Image
from google.colab import files
from IPython.display import display, Video
if not torch.cuda.is_available(): raise RuntimeError('No GPU available. Choose Runtime > Change runtime type > GPU if available.')
print(torch.cuda.get_device_name(0))
uploaded = files.upload()  # Select one approved scene image from your phone/repository.
if not uploaded: raise RuntimeError('Upload one approved scene image.')
input_path = next(iter(uploaded))
image = Image.open(input_path).convert('RGB').resize((512, 768), Image.Resampling.LANCZOS)
display(image)

## Generate one short clip

Edit the prompt to match the scene. Use an image that already shows the character's body if you want full-body movement. Inspect the result before using it.

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video
pipe = LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
prompt = 'Cinematic realistic drama shot, preserve the same person, face, hairstyle, clothing and scene as the input image. Natural subtle full-body movement, gentle breathing, small head turn and believable weight shift, hair and clothing move softly, slow cinematic camera push-in, consistent background, continuous single shot.'
negative = 'face identity change, different clothes, extra people, extra limbs, extra fingers, deformed hands, scene cut, flicker, jitter, text, watermark, blurry'
result = pipe(image=image, prompt=prompt, negative_prompt=negative, width=512, height=768, num_frames=33, num_inference_steps=25, guidance_scale=3.0, generator=torch.Generator(device='cpu').manual_seed(42)).frames[0]
output_path = 'scene_motion_test.mp4'
export_to_video(result, output_path, fps=24)
display(Video(output_path, embed=True))

## Download and add to GitHub

Keep only clips that preserve the character's identity and look natural. Download the clip, then upload it to `assets/episodes/episode_001/motion/scene_001.mp4` (change the scene number as needed). Keep the original approved still image. The GitHub renderer now prefers a matching motion clip and adds the generated voiceover. Free Colab may disconnect or run out of memory.

## Environment repair — restart before continuing

The install cell removes the incompatible `torchao` package and pins a stable Diffusers/Transformers combination. After it finishes, choose **Runtime → Restart session**. Then rerun the GPU check, skip the install cell, and continue with the image upload. This prevents old Python imports from remaining in memory.

In [ ]:
files.download(output_path)